# Exercise 6A. Classification, deep learning, model training

The aim of exercise 6 is to learn how to:
* train deep learning models (MLP)
* use the trained model to predict a whole image
* evaluate the model's accuracy
* see what features mostly contribute to the results

The task is land-use classification, based on 2 datasets: Sentinel-2 11-days mosaic and Google AlphaEarth data.

This exercise is split into two notebooks:
* This for model training
* `06B_deep_classification_prediction_and_evaluation.ipynb` for model evaluation and prediction.

**You need to run this notebook twice, for the 6B exercise to work properly. Change the input dataset for the second run, look for cell marked #TODO**

## Input data

3 raster files with:

* Coordinate system: Finnish ETRS-TM35FIN, EPSG:3067
* Resolution: 20m

### Labels

Multiclass classification raster: 
* 1 - forest
* 2 - fields
* 3 - water
* 0 - everything else

### Data 

**Sentinel2 mosaic**
* Date: 2021-05-22- 2021-05-31
* 10 bands: 'b02', 'b03', 'b04', 'b05', 'b06', 'b07', 'b08', 'b8a', 'b11', 'b12'.
* The reflection values scaled to [0 ... 1].
      
**AlphaEarth**
* 2021
* 64 bands
* Values kept to original -127 to 127. For our tree-based models this is ok, for some other models [de-quantization](https://developers.google.com/earth-engine/guides/aef_on_gcs_readme#de-quantization) should be used.


## Results

A trained MLP model for both datasets.

## Main steps

1) Read data and shape it to suitable form for scikit-learn.
2) Divide the data to training, validation and test datasets using spatial blocks.
3) Undersample to balance the training dataset.
4) Train the model.

The notebook is very similar to 4A shallow classification training, except file names only the model training section is different.

## Imports and paths

In [ ]:
import os, time
from imblearn.under_sampling import RandomUnderSampler
from joblib import dump, load
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.plot import show_hist
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
%matplotlib inline

In [ ]:
# Set folders
user = os.environ.get('USER')
base_folder = os.path.join('/scratch/project_2019932/students', user, 'GeoML')
dataFolder = os.path.join(base_folder,'data', 'raster', 'pixel-wise')
outputBaseFolder= os.path.join(base_folder,'06_deep_classification_MLP')

# Source data local paths
multiclass_classification_file = os.path.join(dataFolder, 'labels.tif')

# Available cores. 
n_jobs = len(os.sched_getaffinity(0))

### TODO, change the input dataset

In [ ]:
# Comment out the first line and uncomment the second line
input_data = "sentinel2"
#input_data = "alphaearth"

In [ ]:
filename = "data_" + input_data + ".tif"
image_file = os.path.join(dataFolder, filename)
image_file

Get the input data. 

* During the CSC course, copy the data to your own folder from Roihu.
* If doing this exercise outside of CSC course, run the [raster data preparations exercise](../02_raster_data_preparation) first.

In [ ]:
! mkdir -p /scratch/project_2019932/raster_data /scratch/project_2019932/students/$USER/GeoML/data/raster
! test -f /scratch/project_2019932/students/$USER/GeoML/data/raster/pixel-wise/labels.tif || cp -R /scratch/project_2019932/raster_data/pixel-wise /scratch/project_2019932/students/$USER/GeoML/data/raster/

## 4.1 Read data and shape it to suitable form for scikit-learn

Read the input dataset with Rasterio and shape it to suitable form for scikit-learn.

### Data

The satellite image has 8 channels (AlphaEarth 64), so rasterio reads it in as 3D data cube.

For scikit-learn we reshape the data to 2D, having in dataframe one row for each pixel. Each pixel has 8 (64) values, one for each band.

In [ ]:
# Read the pixel values from .tif file as dataframe
with rasterio.open(image_file) as image_dataset:
    image_data = image_dataset.read()

# Check shape of input data
print ('Dataframe original shape, 3D: ', image_data.shape)    

Save number of bands for later, to be able to reshape data back to 3D.

In [ ]:
no_bands_in_image = image_data.shape[0]
no_bands_in_image

As a mid-step transponse the axis order, so that the bands are the last. Notice how the dataframe size changes.

In [ ]:
image_data2 = np.transpose(image_data, (1, 2, 0))
# Check again the data shape, now the bands should be last.
print ('Dataframe shape after transpose, 3D: ', image_data2.shape) 

In [ ]:
# Then reshape to 2D.
pixels = image_data2.reshape(-1, no_bands_in_image)
print ('Dataframe shape after transpose and reshape, 2D: ', pixels.shape) 

### Labels
Do the same for labels.

In [ ]:
# For labels only reshape to 1D is enough.
with rasterio.open(multiclass_classification_file) as labels_src:
    labels_data = labels_src.read()
    input_labels = labels_data.reshape(-1)
    print ('Labels shape after reshape, 1D: ', input_labels.shape)

Notice that labels data has only one band.

In [ ]:
labels_data.shape

## 4.2 Divide data to training, validation and test datasets based on spatial location

The neighboring pixels in a raster are highly correlated, i.e. there is spatial autocorrelation. We can try and mitigate this by first splitting the data into spatial blocks and then divide the data into training, validation and test based on the blocks such that data from the whole block is assigned to only one of the sets. 

In [ ]:
_, H, W = image_data.shape
n_blocks_per_side = 6  # Tune this according to the distances

block_h = H // n_blocks_per_side # Divide the number of pixels in height and width into blocks 
block_w = W // n_blocks_per_side
row_idx, col_idx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij') # Assign an index to each block (e.g. 0,0 is the left most upper block)
block_id = (row_idx // block_h) * n_blocks_per_side + (col_idx // block_w) # Flatten the 2D block position to one integer ranging from 0 to 42
block_id

Now we have raster data as a 2D dataframe (1000000, 8) and labels as a flattened vector (1000000,). Let's flatten the block_id array also. 

In [ ]:
groups = block_id.reshape(-1)

Set training, validation and test data ratios, how big part of the pixels is assigned to different sets.

In [ ]:
train_ratio = 0.7
validation_ratio = 0.2
test_ratio = 0.1

X = pixels
y = input_labels

Split into training, validation and test sets using Sklearn GroupShuffleSplit based on the blocks. Let's first separate test set.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=2) 
rest_idx, test_idx = next(gss.split(X, y, groups=groups))

X_rest, X_test = X[rest_idx], X[test_idx]
y_rest, y_test = y[rest_idx], y[test_idx]

... and then training and validation set, keeping the neighboring pixels in the same splits.  

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=validation_ratio/(train_ratio + validation_ratio), random_state=2)
train_idx, val_idx = next(gss.split(X, y, groups=groups))

X_train1, X_validation = X[train_idx], X[val_idx]
y_train1, y_validation = y[train_idx], y[val_idx]

Save the test data for evaluation and prediction.

In [ ]:
test_data_file = input_data + "_test_data.npz"
np.savez(test_data_file,  X_test=X_test, y_test=y_test)

## 4.3 Resample to balance the dataset

The classes are very imbalanced in the dataset, so undersample the majority classes in the training set, so that all classes are represented about similar number of pixels. 
Notice that validation and test set keep the original class-distribution. Remember: 1 - forest, 2 - fields, 3 - water, 0 - everything else

In [ ]:
a = show_hist(labels_data, label='Classes')

In [ ]:
rus = RandomUnderSampler(random_state=63)
x_train, y_train = rus.fit_resample(X_train1, y_train1)   
print ('Dataframe shape after undersampling of majority classes, 2D: ', x_train.shape)

*How many pixels of different classes are included in training dataset?*

Notice that we lost a lot of pixels at this point, in real cases that may be undesired. See [imbalanced-learn user guide](https://imbalanced-learn.org/stable/user_guide.html#user-guide) for other options.

In [ ]:
print('Labels before splitting:           ', np.unique(input_labels, return_counts=True)[1])
print('Training data before undersampling:', np.unique(y_train1, return_counts=True)[1])
print('Training data after undersampling: ', np.unique(y_train, return_counts=True)[1])
print('Validation data:                   ', np.unique(y_validation, return_counts=True)[1])
print('Test data:                         ', np.unique(y_test, return_counts=True)[1])

## 4.4 Model training
### Functions for training and estimating the models

Similar functions will be used by different algorithms. Here the functions are only defined, they will be used later.

### Train the model

In [ ]:
def trainModel(x_train, y_train, clf, classifierName):
    start_time = time.time()    
    clf.fit(x_train, y_train)
    print('Model training took: ', round((time.time() - start_time), 2), ' seconds')
    
    # Save the model to a file
    modelFilePath = os.path.join(outputBaseFolder, ('model_' + input_data + '_' + classifierName + '.sav'))
    dump(clf, modelFilePath) 
    return clf

In [ ]:
classifierName = 'mlp'
# Initialize the MLP classifier and give the hyperparameters.
mlp_classifier = MLPClassifier(hidden_layer_sizes=(128,64,32,16), activation='relu', 
                                          solver='adam', early_stopping=True, batch_size=256, random_state=63,
                                          validation_fraction=validation_ratio, n_iter_no_change=10)
mlp_classifier = trainModel(x_train, y_train, mlp_classifier, classifierName)

*Please, be patient, this takes a moment*
 - `hidden_layer_sizes` - how many and how big hidden layers to use, try to modify the number of layers or size of layers.
 - `Adam optimizer`, often used, scikit-learn supports also a few others

Plot the train loss of the latest model fitting

In [ ]:
plt.plot(mlp_classifier.loss_curve_)
plt.show()

Plot the validation accuracy

In [ ]:
plt.plot(mlp_classifier.validation_scores_)
plt.show()

In [ ]:
print("Accuracy of the model with training data:", mlp_classifier.score(x_train, y_train))

*Feel free to modify some of the hyperparameters above to get better results.*
And then see with test data, if the modifications help also for previously unseen data.

### Estimate the model

Model is estimated with validation data first and later with test data. Both confusion matrix and classification report are generated.

In [ ]:
def estimateModel(clf, x_test, y_test):
    test_predictions = clf.predict(x_test)
    print('Classification report: \n', classification_report(y_test, test_predictions))
    ConfusionMatrixDisplay.from_predictions(y_test, test_predictions, normalize='true', cmap=plt.cm.Blues)

In [ ]:
estimateModel(mlp_classifier, X_test, y_test) #Test data